# Data Organize — clean & feature-prep

Read the raw race table from `data_loading` (`data/raw/races.parquet`), clean it, add a few leakage-safe features, and save `data/processed/features.parquet` for `model_training`.

**Leakage rule:** a feature for a race may only use information from *earlier* races. That's why every form feature does `groupby(...).shift(1)` before rolling.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from f1pred import paths                          # ROOT/DATA/RAW/... path helpers

# Read the tidy race table that data_loading saved (one row per driver per race).
races = pd.read_parquet(paths.RAW / "races.parquet")
# Put rows in chronological order and give them a clean 0..N index.
# (reset_index(drop=True) throws away the old shuffled index instead of keeping it as a column.)
races = races.sort_values(["season", "round"]).reset_index(drop=True)

# Sanity: we actually loaded rows, and the columns we depend on below are present.
assert len(races) > 0, "races.parquet is empty -- run the data_loading notebook first"
missing = {"season", "round", "DriverId"} - set(races.columns)
assert not missing, f"races.parquet is missing expected columns: {missing}"
races.head()

,DriverId,Abbreviation,TeamId,grid_position,race_position,ClassifiedPosition,points,status,quali_position,q1_time,q2_time,q3_time,best_quali,season,round,event
0,leclerc,LEC,ferrari,1.0,1.0,1,26.0,Finished,1.0,0 days 00:01:31.471000,0 days 00:01:30.932000,0 days 00:01:30.558000,0 days 00:01:30.558000,2022,1,Bahrain Grand Prix
1,sainz,SAI,ferrari,3.0,2.0,2,18.0,Finished,3.0,0 days 00:01:31.567000,0 days 00:01:30.787000,0 days 00:01:30.687000,0 days 00:01:30.687000,2022,1,Bahrain Grand Prix
2,hamilton,HAM,mercedes,5.0,3.0,3,15.0,Finished,5.0,0 days 00:01:32.285000,0 days 00:01:31.048000,0 days 00:01:31.238000,0 days 00:01:31.048000,2022,1,Bahrain Grand Prix
3,russell,RUS,mercedes,9.0,4.0,4,12.0,Finished,9.0,0 days 00:01:32.269000,0 days 00:01:31.252000,0 days 00:01:32.216000,0 days 00:01:31.252000,2022,1,Bahrain Grand Prix
4,kevin_magnussen,MAG,haas,7.0,5.0,5,10.0,Finished,7.0,0 days 00:01:31.955000,0 days 00:01:31.461000,0 days 00:01:31.808000,0 days 00:01:31.461000,2022,1,Bahrain Grand Prix


## 1. Clean

Fix the known gotchas and define the prediction target (finishing order).

In [ ]:
# Work on a copy so re-running this cell never mutates the freshly-loaded `races`.
df = races.copy()

# --- drop broken races -----------------------------------------------------
# A handful of 2025-2026 rounds came back from FastF1 with an EMPTY DriverId and
# no results. Worse, the quali merge in data_loading joined those empty-string
# ids together and exploded them into hundreds of duplicate NaN rows (one race
# had 484). Keep only rows that have a real driver id -> those broken races drop
# out entirely, leaving ~90 clean races.
df = df[df["DriverId"].astype(str).str.strip() != ""].copy()

# --- grid position fix -----------------------------------------------------
# FastF1 encodes a pit-lane start as grid position 0.0. Left as-is, the model
# would read "0" as even better than pole (1). Step 1: turn those 0s into NaN...
df["grid_start"] = df["grid_position"].replace(0, np.nan)
# ...Step 2: fill each NaN with (the worst real grid slot in THAT race) + 1,
# i.e. drop pit-lane starters to the very back of their own race's grid.
#   groupby([...])["grid_start"].transform("max") -> the max grid within each
#   (season, round), broadcast back onto every row of that race so shapes align.
df["grid_start"] = df["grid_start"].fillna(
    df.groupby(["season", "round"])["grid_start"].transform("max") + 1
)

# --- did they actually finish? --------------------------------------------
# ClassifiedPosition is a STRING: "1".."20" for classified finishers, or letter
# codes like "R" (retired) / "D" (disqualified). fullmatch(r"\d+") is True only
# when the whole value is digits -> a real finish. .fillna(False) handles NaN.
df["finished"] = df["ClassifiedPosition"].str.fullmatch(r"\d+").fillna(False)

# --- qualifying pace as a plain number ------------------------------------
# best_quali is a timedelta (e.g. 0:01:30.558000). Models need a float, so
# convert the lap to seconds -> 90.558.
df["quali_seconds"] = df["best_quali"].dt.total_seconds()

# --- prediction target: finishing order -----------------------------------
# n_drivers = how many cars were in each race (group size), broadcast to every
# row of that race so the fillna below has a per-race number to use.
df["n_drivers"] = df.groupby(["season", "round"])["DriverId"].transform("size")
# target = finishing position. A DNF has race_position = NaN, so we fill it with
# n_drivers -> the driver sorts to the back of that race instead of vanishing.
df["target"] = df["race_position"].fillna(df["n_drivers"])

# Sanity: every row now has a grid slot and a target (nothing silently NaN).
# The message prints the offending counts so a failure tells you WHAT is wrong.
bad_grid = int(df["grid_start"].isna().sum())
bad_target = int(df["target"].isna().sum())
assert bad_grid == 0 and bad_target == 0, (
    f"still NaN after cleaning -> grid_start: {bad_grid}, target: {bad_target}. "
    "Likely broken/empty rows that slipped past the DriverId drop above."
)

## 2. Features

A handful of obvious, leakage-safe signals. Form features look only at past races (`shift(1)` first), so a driver's/team's debut race is `NaN` — which is correct.

In [ ]:
# --- driver recent form ----------------------------------------------------
# Sort so each driver's races run oldest -> newest; the rolling window below
# assumes rows are in time order WITHIN each group.
df = df.sort_values(["DriverId", "season", "round"])
# For each driver, build a leakage-safe "recent form" number:
#   .shift(1)                  -> drop the CURRENT race, so we only ever look backward
#   .rolling(5, min_periods=1) -> a window of up to the last 5 races (fewer is OK early on)
#   .mean()                    -> their average finishing position across that window
# transform(...) returns a Series aligned to df's index, so it drops straight into a column.
df["driver_form"] = df.groupby("DriverId")["target"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

# --- team recent form ------------------------------------------------------
# Identical idea, but grouped by TeamId -> how the car/constructor has been doing.
df = df.sort_values(["TeamId", "season", "round"])
df["team_form"] = df.groupby("TeamId")["target"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

# Now that the form columns are built, restore normal race order + a clean index.
df = df.sort_values(["season", "round"]).reset_index(drop=True)

# --- qualifying gap to pole -----------------------------------------------
# Within each race, subtract the fastest quali lap (pole time) from every driver's
# lap -> seconds behind pole. transform("min") broadcasts that race's pole time to
# all its rows. The pole sitter gets 0.0; everyone else is positive.
df["quali_gap"] = df["quali_seconds"] - df.groupby(["season", "round"])["quali_seconds"].transform("min")

# --- grid penalty ----------------------------------------------------------
# Places lost between qualifying and the actual start (grid penalties / pit starts).
# Positive = started further back than they qualified.
df["grid_penalty"] = df["grid_start"] - df["quali_position"]

# Sanity: the four feature columns exist...
missing = {"driver_form", "team_form", "quali_gap", "grid_penalty"} - set(df.columns)
assert not missing, f"feature columns were not built: {missing}"
# ...and a driver's very first race has NaN form (no past to average). If this is
# ever False, the shift(1) leakage guard isn't working (form would peek at the present).
assert df["driver_form"].isna().any(), (
    "expected some NaN in driver_form (debut races have no prior form) -- "
    "the shift(1) leakage guard may be broken"
)

## 3. Save

Write the model-ready table for `model_training`.

In [ ]:
out_dir = paths.DATA / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / "features.parquet"
df.to_parquet(out, index=False)

assert out.exists(), f"failed to write {out}"
print("wrote", out, "->", df.shape)

wrote C:\Users\tusha\Documents\Project\f1-pred-modal\data\processed\features.parquet -> (1810, 25)
